In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("1. Initializing Embedding model (converting text to vectors)...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. The data chunks from our previous experiment (Pipeline B)
formula_aware_chunks = [
    r"We can now apply Newton's second law of motion to this system. The fundamental principle states that the net force is equal to mass times acceleration.Therefore, the equation of motion is:\n$$ \sum \vec{F} = m \vec{a} $$",
    r"Where m is the mass of the object and a is the acceleration vector. If we consider the force of gravity, the gravitational force between two masses is defined as:\n$$ F_g = G \frac{m_1 m_2}{r^2} $$",
    r"This equation is crucial for understanding planetary orbits. Now, let's move on to the concept of kinetic energy, which is given by the formula half mass times velocity squared."
]

print("2. Storing chunks into Chroma Vector Database...")
# Create the vector database and insert our chunks
vector_db = Chroma.from_texts(
    texts=formula_aware_chunks,
    embedding=embedding_model,
    collection_name="physics_knowledge_base"
)

print("3. Database is ready! Let's perform a retrieval simulation.")
print("-" * 50)

# 4. Query simulation from your evaluation dataset
# Imagine this is a question from your MMLU CSV file
query = "What is the mathematical equation for the gravitational force between two masses?"

print(f"QUESTION: {query}\n")

# Retrieve the top 1 chunk that is most semantically similar to the query
retrieved_docs = vector_db.similarity_search(query, k=1)

print("SEARCH RESULT FROM DATABASE (Context to be fed to the LLM):")
print(retrieved_docs[0].page_content)

1. Initializing Embedding model (converting text to vectors)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2. Storing chunks into Chroma Vector Database...
3. Database is ready! Let's perform a retrieval simulation.
--------------------------------------------------
QUESTION: What is the mathematical equation for the gravitational force between two masses?

SEARCH RESULT FROM DATABASE (Context to be fed to the LLM):
Where m is the mass of the object and a is the acceleration vector. If we consider the force of gravity, the gravitational force between two masses is defined as:\n$$ F_g = G \frac{m_1 m_2}{r^2} $$


In [3]:
import os
import getpass
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

print("1. Setting up Groq API Key...")
# Securely prompt for the API key so it is not hardcoded in the script
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key (gsk_...): ")

print("2. Connecting to LLM (Llama-3.1 via Groq)...")
# Initialize the newer Llama-3.1 model
# The previous model was decommissioned, so we use 'llama-3.1-8b-instant'
llm = ChatGroq(
    model="llama-3.1-8b-instant", 
    temperature=0.0 
)

print("3. Building the Prompt (Custom Instructions for the AI)...")
# Create a template that strictly bounds the AI to only use our retrieved context
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a brilliant college physics professor. 
Answer the physics question based ONLY on the following context. If the answer is not in the context, say "I don't know based on the provided context." Do not use outside knowledge.

CONTEXT:
{context}

QUESTION: 
{question}

ANSWER:"""
)

print("4. Generating the Final Answer...")
# Extract the text context that our vector database successfully retrieved earlier
context_text = retrieved_docs[0].page_content

# Format the prompt by injecting the context and the question
final_prompt = prompt_template.format(context=context_text, question=query)

# Ask the LLM to generate the response
response = llm.invoke(final_prompt)

print("\n" + "="*50)
print("QUESTION: ", query)
print("-" * 50)
print("LLM ANSWER (Llama-3.1):")
print(response.content)
print("="*50)

1. Setting up Groq API Key...
2. Connecting to LLM (Llama-3.1 via Groq)...
3. Building the Prompt (Custom Instructions for the AI)...
4. Generating the Final Answer...

QUESTION:  What is the mathematical equation for the gravitational force between two masses?
--------------------------------------------------
LLM ANSWER (Llama-3.1):
The mathematical equation for the gravitational force between two masses is:

$$ F_g = G \frac{m_1 m_2}{r^2} $$
